# Visualize evaluation results

In [ ]:
from matplotlib import pyplot as plt
import ast
from pathlib import Path

import config

## Read results from files

In [ ]:
# Read results from files for laat
infids = {}
infid_key = 'mean_infid'
times = {}
time_key = 'mean_time'

for model_name in config.MODELS:
    infids[model_name] = {}
    times[model_name] = {}
    # Read ixg results
    method_name = 'ixg'
    file_name = Path(f'results/{model_name}_{method_name}_thresh_05_seed_10.txt')
    if file_name.is_file():
        with open(file_name) as f:
            results = ast.literal_eval(f.read())
            infids[model_name][method_name] = results[infid_key]
            times[model_name][method_name] = results[time_key]
    # Read ig results
    method_name = 'ig'
    for n_steps in config.N_STEPS:
        file_name = Path(f'results/{model_name}_{method_name}_thresh_05_seed_10_nsteps_{n_steps}.txt')
        if file_name.is_file():
            with open(file_name) as f:
                results = ast.literal_eval(f.read())
                method_name_full = method_name + str(n_steps)
                infids[model_name][method_name_full] = results[infid_key]
                times[model_name][method_name_full] = results[time_key]
    # Read shap results
    method_name = 'shap'
    for n_samples in config.N_SAMPLES:
        file_name = Path(f'results/{model_name}_{method_name}_thresh_05_seed_10_nsamples_{n_samples}.txt')
        if file_name.is_file():
            with open(file_name) as f:
                results = ast.literal_eval(f.read())
                method_name_full = method_name + str(n_samples)
                infids[model_name][method_name_full] = results[infid_key]
                times[model_name][method_name_full] = results[time_key]

## Plot results

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10,8))
width = 0.75    # Bar width
alpha = 0.5     # Opacity
for i, model_name in enumerate(infids.keys()):
    method_names = list(infids[model_name].keys())
    # Plot infidelities
    infids_values = list(infids[model_name].values())
    axs[0, i].barh(method_names, infids_values, width, facecolor='b', alpha=0.5)
    axs[0, i].set_xlim([0., 1.2 * max(infids_values)])
    axs[0, i].set_title(f"Infidelity of feature attributions on {model_name}")
    axs[0, i].set_xlabel("Infidelity (SD=1.0)")
    for idx, v in enumerate(infids_values):
        axs[0, i].text(v + 0.02 * v, idx, str(v), color='black')
    # Plot times
    times_values = list(times[model_name].values())
    axs[1, i].barh(method_names, times_values, width, facecolor='b', alpha=0.7)
    axs[1, i].set_xlim([0., 1.2 * max(times_values)])
    axs[1, i].set_title(f"Runtime of feature attributions on {model_name}")
    axs[1, i].set_xlabel("Runtime (in s)")
    for idx, v in enumerate(times_values):
        axs[1, i].text(v + 0.02 * v, idx, str(round(v, 2)), color='black')

fig.tight_layout()

In [ ]:
# Save results plot
thresh = str(config.THRESHOLD).replace('.', '')
fig.savefig(f"results/barplot_thresh_{thresh}_seed_{config.SEED}.eps", format="eps")